# Baseline Training: YOLOv11-OBB on RSDD-SAR

**What this notebook does:**
Trains a rotated bounding-box object detector (`yolo11n-obb`) on the RSDD-SAR ship dataset,
then evaluates it on the full test set plus inshore/offshore subsets.

**Dataset:** RSDD-SAR — 7,000 SAR chip images (~512×512 px, ~3 m resolution),
10,263 annotated ship instances. Pre-converted to YOLO-OBB polygon format locally.

**What gets produced:**
- `runs/rsdd_baseline/v1_n/weights/best.pt` — best checkpoint by validation mAP
- `runs/rsdd_baseline/v1_n/weights/last.pt` — final epoch checkpoint
- Confusion matrix, P/R curves, loss plots in `runs/rsdd_baseline/v1_n/`
- Evaluation results saved to Drive so they survive session end

**Expected runtime:** 1–3 hours on T4 GPU; 30–60 min on A100.

**Before running:**
1. Set Runtime → Change runtime type → T4 GPU (or A100 if available)
2. Upload the YOLO-converted dataset to Google Drive at:
   `/MyDrive/sar-dark-ship/rsdd_yolo/` (conversion runs locally: `python -m src.data.rsdd_to_yolo --copy-images`)
3. Run cells in order — don't skip the GPU check

## 1. Environment Setup

Colab has PyTorch and most ML libraries pre-installed, but Ultralytics (the YOLOv11 library)
needs to be explicitly installed. This takes ~30 seconds.

In [ ]:
# Install Ultralytics (provides yolo11n-obb weights + training loop)
!pip install -q ultralytics

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → GPU (T4 or A100) "
        "and re-run from the top."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

from ultralytics import YOLO
print("Ultralytics imported OK")

## 2. Load Dataset from Drive

The converted dataset lives in Google Drive. We copy it to Colab's local `/content/` storage
because reading from Drive during training is slow (network I/O per batch).

**Expected Drive structure** (produced by the local converter):
```
MyDrive/sar-dark-ship/rsdd_yolo/
├── dataset.yaml          ← YOLO config (auto-generated by converter)
├── images/
│   ├── train/            ← 5,000 images
│   └── test/             ← 2,000 images (used as val during training)
└── labels/
    ├── train/            ← 5,000 .txt label files
    ├── test/             ← 2,000 .txt label files
    ├── test_inshore/     ← 159 label files  (subset of test)
    └── test_offshore/    ← 1,841 label files (subset of test)
```

The copy takes ~5 minutes — it runs once per session. If you restart and `/content/rsdd_yolo` already exists, it's skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, time

# ── Adjust SRC if you uploaded to a different Drive path ──
SRC = "/content/drive/MyDrive/sar-dark-ship/rsdd_yolo"
DST = "/content/rsdd_yolo"

if os.path.exists(DST):
    print(f"{DST} already exists — skipping copy (delete it to force a fresh copy)")
else:
    print(f"Copying {SRC} → {DST} ...")
    t0 = time.time()
    shutil.copytree(SRC, DST)
    print(f"Done in {time.time()-t0:.0f}s")

# Verify structure
for split in ['train', 'test']:
    n_img = len(os.listdir(f"{DST}/images/{split}"))
    n_lbl = len(os.listdir(f"{DST}/labels/{split}"))
    print(f"  {split}: {n_img} images, {n_lbl} labels")

## 3. Dataset Config

The converter already wrote `dataset.yaml` into the output dir. We overwrite it here
with the Colab-local absolute path so YOLO can find the files.

Note: RSDD-SAR has no separate val split. We use `test` for both val (during training)
and final evaluation. This is standard practice for datasets without an explicit val set —
it means the val mAP during training is the same as the held-out test mAP, which is fine
for a research baseline (we just can't use it for hyperparameter tuning).

In [ ]:
import yaml

data_cfg = {
    "path": "/content/rsdd_yolo",
    "train": "images/train",
    "val":   "images/test",   # no separate val split — use test
    "test":  "images/test",
    "names": {0: "ship"},
}

yaml_path = "/content/rsdd_yolo/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"Wrote {yaml_path}")
print(open(yaml_path).read())

## 4. Hyperparameters

**Model: `yolo11n-obb`** — the nano variant of YOLOv11 for oriented bounding boxes.
"Nano" means ~2.7M parameters — fast to train, fits in T4 VRAM with batch 16, good for a
baseline. If results are weak we can step up to `yolo11s-obb` (small, ~9M params).

**512×512 images** — RSDD-SAR chips are natively 512×512, so no resizing overhead.

**patience=20** — early stopping: if val mAP doesn't improve for 20 consecutive epochs,
training stops. This prevents wasting GPU time once the model has converged.

## 5. Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-obb.pt")   # downloads pretrained weights on first run (~6 MB)

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=512,
    batch=16,
    device=0,            # GPU 0
    patience=20,         # early stopping
    project="runs/rsdd_baseline",
    name="v1_n",
    save=True,
    plots=True,
)

print("Training complete.")
print(f"Best weights: {results.save_dir}/weights/best.pt")

## 6. Evaluation

We evaluate on three splits separately:
- **test** — full 2,000-image held-out test set (overall in-domain performance)
- **test_inshore** — 159 images from inshore scenes (harbours, coastal)
- **test_offshore** — 1,841 images from offshore scenes (open ocean)

Inshore/offshore results tell us whether the model is stronger in one environment.
SAR imagery of inshore areas is harder (clutter, port infrastructure), so we expect
lower recall there.

**Implementation note:** `model.val()` accepts a `data` YAML path. We generate
temporary per-split YAMLs pointing to the right images and labels directories,
then call `.val()` three times.

In [ ]:
import yaml, os
from ultralytics import YOLO

# Load best weights
best_weights = f"runs/rsdd_baseline/v1_n/weights/best.pt"
model = YOLO(best_weights)

def make_split_yaml(split_name: str, base_dir: str = "/content/rsdd_yolo") -> str:
    """Write a minimal YOLO yaml for a single split and return its path."""
    cfg = {
        "path": base_dir,
        "val":  f"images/{split_name if split_name == 'test' else 'test'}",
        "names": {0: "ship"},
    }
    # For inshore/offshore: images are in images/test, labels in labels/<split>
    # YOLO infers label paths from image paths by replacing 'images' with 'labels'.
    # Since inshore/offshore images ARE the test images, we point val at images/test
    # but need a way to filter to only the right labels.
    #
    # Workaround: create a symlinked images dir containing only the split's images.
    if split_name in ('test_inshore', 'test_offshore'):
        split_img_dir = f"/content/rsdd_yolo/images/{split_name}"
        os.makedirs(split_img_dir, exist_ok=True)
        # Symlink images that have a label in this split's labels dir
        label_dir = f"/content/rsdd_yolo/labels/{split_name}"
        for lbl_file in os.listdir(label_dir):
            stem = os.path.splitext(lbl_file)[0]
            src_img = f"/content/rsdd_yolo/images/test/{stem}.jpg"
            dst_img = f"{split_img_dir}/{stem}.jpg"
            if os.path.exists(src_img) and not os.path.exists(dst_img):
                os.symlink(src_img, dst_img)
        cfg["val"] = f"images/{split_name}"

    yaml_path = f"/content/rsdd_yolo/eval_{split_name}.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return yaml_path


results_by_split = {}
for split in ["test", "test_inshore", "test_offshore"]:
    print(f"\n{'='*50}")
    print(f"Evaluating: {split}")
    print(f"{'='*50}")
    yaml_path_split = make_split_yaml(split)
    metrics = model.val(
        data=yaml_path_split,
        imgsz=512,
        device=0,
        split="val",
        name=f"eval_{split}",
        project="runs/rsdd_baseline",
    )
    results_by_split[split] = {
        "mAP50":    round(float(metrics.box.map50),  4),
        "mAP50_95": round(float(metrics.box.map),    4),
        "precision": round(float(metrics.box.mp),    4),
        "recall":    round(float(metrics.box.mr),    4),
    }

# Print results table
print("\n" + "="*65)
print(f"{'Split':<20} {'mAP@0.5':>8} {'mAP@0.5:0.95':>12} {'Precision':>10} {'Recall':>8}")
print("-"*65)
for split, m in results_by_split.items():
    print(f"{split:<20} {m['mAP50']:>8.4f} {m['mAP50_95']:>12.4f} {m['precision']:>10.4f} {m['recall']:>8.4f}")
print("="*65)

# Save results to a text file so we can read them back
import json
with open("runs/rsdd_baseline/v1_n/eval_results.json", "w") as f:
    json.dump(results_by_split, f, indent=2)
print("\nResults saved to runs/rsdd_baseline/v1_n/eval_results.json")

## 7. Save to Drive

Colab sessions expire. Copy weights and eval results to Drive now so they persist.

After this, download `model_v1_baseline.pt` to your Mac for Day 5 (HF Hub upload).

In [ ]:
import shutil, os

DRIVE_OUT = "/content/drive/MyDrive/sar-dark-ship"
os.makedirs(DRIVE_OUT, exist_ok=True)

# Save best weights
best_weights = "runs/rsdd_baseline/v1_n/weights/best.pt"
shutil.copy(best_weights, f"{DRIVE_OUT}/model_v1_baseline.pt")
print(f"Weights saved to {DRIVE_OUT}/model_v1_baseline.pt")

# Save eval results
shutil.copy(
    "runs/rsdd_baseline/v1_n/eval_results.json",
    f"{DRIVE_OUT}/eval_results_v1.json"
)
print(f"Eval results saved to {DRIVE_OUT}/eval_results_v1.json")

# Copy full run directory (loss curves, confusion matrix, etc.)
run_dst = f"{DRIVE_OUT}/runs_v1"
if not os.path.exists(run_dst):
    shutil.copytree("runs/rsdd_baseline/v1_n", run_dst)
    print(f"Full run saved to {run_dst}")
else:
    print(f"{run_dst} already exists — skipping run copy")

## 8. Next Steps

Once training is done and weights are on Drive:

1. **Download weights to Mac:** In Drive, right-click `model_v1_baseline.pt` → Download.  
   Or use `huggingface_hub` to push directly from Colab — see Day 5 instructions.

2. **Day 4 (eval harness):** The local `src/eval/obb_metrics.py` module implements
   rotated mAP using Shapely polygon IoU. You can run this on the Colab outputs
   to cross-check the Ultralytics mAP numbers.

3. **Day 5 (model card + HF Hub upload):**
   ```python
   from huggingface_hub import HfApi
   api = HfApi()
   api.upload_file(
       path_or_fileobj="model_v1_baseline.pt",
       path_in_repo="model_v1_baseline.pt",
       repo_id="tejassnaikk/rsdd-yolo11n-obb-v1",
       repo_type="model",
   )
   ```

4. **Update `reports/metrics.md`** with the numbers from `eval_results_v1.json`.

5. **Week 2:** Sentinel-1 GRD download + preprocessing. Register at
   [dataspace.copernicus.eu](https://dataspace.copernicus.eu) if you haven't already.